# TF-IDF Content-Based Course Recommender

This notebook builds a content-based recommendation model for the Online Course Recommender System project.

The goal is to recommend courses that are textually similar to a selected course. Instead of using ratings, clicks, purchases, or other user behavior, this approach uses course information that is already available in the dataset.

This TF-IDF recommender serves as the baseline recommendation model for the project. Later, its recommendations can be compared with the KNN recommender to evaluate how different similarity-based approaches perform.

## 1. Introduction

### What is content-based recommendation?

A content-based recommender suggests items that are similar to an item the user already knows or likes. In this project, each course is represented using its textual features, such as the course title, subject, and level.

If a user is interested in a beginner Python course, a content-based model should recommend other courses with similar words, topics, and difficulty levels.

### Why use TF-IDF?

Course descriptions and titles are text, but machine learning models need numerical input. TF-IDF converts text into numerical vectors while giving more importance to words that are meaningful within a course and less importance to words that appear too often across many courses.

This makes TF-IDF useful for course recommendation because it highlights distinctive terms such as `python`, `excel`, `trading`, `javascript`, or `guitar`.

### Why use cosine similarity?

After TF-IDF converts each course into a vector, cosine similarity measures how close two course vectors are in direction. A higher cosine similarity means the courses use similar terms and are likely to cover related content.

Cosine similarity is especially useful for text because it focuses on the pattern of words rather than only the length of the text.

## 2. Load Processed Dataset

The processed dataset is loaded from `data/processed/processed_udemy_courses.csv`.

This notebook expects the preprocessing notebook to have already created a `combined_features` column. That column contains the textual representation used for recommendation.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
DATA_PATHS = [
    Path("data/processed/processed_udemy_courses.csv"),
    Path("../data/processed/processed_udemy_courses.csv"),
]

DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find processed_udemy_courses.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

The dataset shape shows how many courses and columns are available after preprocessing. The first rows help confirm that the data loaded correctly.

In [ ]:
important_columns = [
    "course_id",
    "course_title",
    "subject",
    "level",
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
    "combined_features",
]

df[important_columns].head()

In [ ]:
if "combined_features" in df.columns:
    print("combined_features column found.")
else:
    raise ValueError("combined_features column is missing. Run the preprocessing notebook first.")

df["combined_features"].isna().sum()

## 3. Feature Inspection

Before building the recommender, it is important to inspect the text that will be used to represent each course.

The `combined_features` field is useful because it combines multiple course attributes into one searchable text field. This allows the recommender to compare courses using topic words, subject categories, and level information together.

In [ ]:
sample_features = df[["course_title", "subject", "level", "combined_features"]].sample(
    5, random_state=42
)

sample_features

Each row now has a text profile. Courses with similar profiles should receive higher similarity scores.

For example, two courses about web development may share words such as `html`, `css`, `javascript`, or `web development`. Two finance courses may share words such as `investment`, `trading`, `stocks`, or `financial`.

## 4. TF-IDF Vectorization

TF-IDF stands for Term Frequency-Inverse Document Frequency.

### Term Frequency (TF)

Term Frequency measures how often a word appears in a document. In this notebook, each course's `combined_features` text is treated as one document.

If the word `python` appears in a course profile, that course receives a higher value for the `python` feature.

### Inverse Document Frequency (IDF)

Inverse Document Frequency reduces the weight of words that appear in many courses. Very common words are usually less useful for distinguishing one course from another.

For example, a word that appears in almost every course is less informative than a word that appears in only a smaller group of related courses.

### Why TF-IDF creates useful numerical vectors

TF-IDF transforms text into a matrix of numbers. Each row represents a course, and each column represents a term from the vocabulary. These vectors can then be compared mathematically.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["combined_features"].fillna(""))

print(f"TF-IDF matrix shape: {tfidf_matrix.shape[0]} courses x {tfidf_matrix.shape[1]} terms")

In [ ]:
feature_names = tfidf.get_feature_names_out()
feature_names[:20]

The TF-IDF matrix has one row per course. The columns represent the vocabulary learned from `combined_features`.

## 5. Cosine Similarity

Cosine similarity measures the angle between two vectors.

- A score close to `1` means two courses are very similar.
- A score close to `0` means two courses are not very similar.

Cosine similarity is suitable for recommendation because course text can vary in length. The method compares the direction of the vectors, which captures whether courses use similar important terms.

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

The similarity matrix compares every course with every other course. The value at row `i`, column `j` is the similarity score between course `i` and course `j`.

## 6. Recommendation Function

The function below takes a course title and returns the top `n` most similar courses.

Steps:

1. Find the selected course in the dataset.
2. Retrieve its similarity scores against all other courses.
3. Sort courses by similarity score from highest to lowest.
4. Remove the selected course itself.
5. Return the top `n` recommendations.

In [ ]:
title_to_index = pd.Series(df.index, index=df["course_title"].str.lower())


def recommend_courses(course_title, n=5):
    """Return the top n courses most similar to the selected course title."""
    normalized_title = course_title.lower().strip()

    if normalized_title not in title_to_index:
        matches = df[df["course_title"].str.lower().str.contains(normalized_title, na=False, regex=False)]
        if matches.empty:
            raise ValueError(f"Course title not found: {course_title}")
        selected_index = matches.index[0]
        print(f"Using matched course: {df.loc[selected_index, 'course_title']}")
    else:
        selected_index = title_to_index[normalized_title]
        if isinstance(selected_index, pd.Series):
            selected_index = selected_index.iloc[0]

    similarity_scores = list(enumerate(cosine_sim[selected_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = [score for score in similarity_scores if score[0] != selected_index]
    top_scores = similarity_scores[:n]

    recommended_indices = [index for index, score in top_scores]
    recommendation_table = df.loc[
        recommended_indices,
        ["course_title", "subject", "level", "num_subscribers", "num_reviews", "price"],
    ].copy()
    recommendation_table["similarity_score"] = [round(score, 3) for index, score in top_scores]

    return recommendation_table.reset_index(drop=True)

### Recommendation Function Edge Cases

If a course title is not found, the function raises an error because the model can only recommend from courses already present in the dataset. If duplicate titles exist, the function uses the first matching course. For much larger datasets, computing a full course-by-course similarity matrix may become memory-intensive, so a more scalable nearest-neighbor search approach may be needed.

## 7. Example Recommendations

The following examples test the recommender with different course categories:

- Python-related course
- Business/finance course
- Web development course

The results are displayed as tables so the recommended course titles, subjects, levels, and similarity scores can be inspected clearly.

In [ ]:
python_course = "web programming with python"

print(f"Recommendations for: {python_course}")
recommend_courses(python_course, n=5)

In [ ]:
business_course = "ultimate investment banking course"

print(f"Recommendations for: {business_course}")
recommend_courses(business_course, n=5)

In [ ]:
web_development_course = "learn complete web development from scratch"

print(f"Recommendations for: {web_development_course}")
recommend_courses(web_development_course, n=5)

## 8. Strengths and Limitations

### Strengths

- Interpretable: recommendations are based on visible course text.
- Simple: the model is easy to understand and implement.
- Works without user interaction data: no ratings, clicks, enrollments, or purchase history are required.
- Useful starting point: content-based recommendations can be created as soon as item metadata is available.

### Limitations

- Depends on available text: if course text is short or incomplete, recommendations may be weaker.
- May over-recommend similar content: the model can suggest courses that are too close to the selected course.
- No user behavior information: the model does not learn from what users actually prefer, ignore, or purchase.
- Vocabulary-based: TF-IDF matches words directly and does not deeply understand meaning or context.

## 9. Conclusions

This notebook created a complete content-based recommender using TF-IDF and cosine similarity.

TF-IDF converts each course's `combined_features` text into a numerical vector. Words that are important to a specific course receive stronger weights, while very common words receive lower weights.

Cosine similarity compares those TF-IDF vectors and identifies courses with similar textual profiles. The recommendation function uses these similarity scores to return the top courses most related to a selected course.

Within the overall Online Course Recommender System project, this notebook provides a clear baseline model. It is especially useful when user interaction data is unavailable and course metadata is the main source of information.